In [ ]:
!pip install ib_insync pandas numpy sklearn matplotlib seaborn pytz

In [ ]:
import pandas as pd
import numpy as np
from ib_insync import IB, Stock
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import matplotlib.pyplot as plt
import pickle
import pytz

In [ ]:
ib = IB()
ib.connect('127.0.0.1', 7497, clientId=1)

In [ ]:
symbols = ["SPY", "AAPL", "NVDA", "MSFT", "AMZN", "META", "TSLA", "GOOGL", "AVGO", "BRK.B"]

In [ ]:
def fetch_daily_data(symbol, duration='2 Y'):
    contract = Stock(symbol, 'SMART', 'USD')
    ib.qualifyContracts(contract)
    bars = ib.reqHistoricalData(contract, endDateTime='', durationStr=duration,
                                barSizeSetting='1 day', whatToShow='TRADES', useRTH=True)
    df = pd.DataFrame(bars)
    df['symbol'] = symbol
    return df

In [ ]:
def fetch_intraday_data(symbol, duration='2 Y', bar_size='30 mins'):
    contract = Stock(symbol, 'SMART', 'USD')
    ib.qualifyContracts(contract)
    bars = ib.reqHistoricalData(contract, endDateTime='', durationStr=duration,
                                barSizeSetting=bar_size, whatToShow='TRADES', useRTH=False)
    df = pd.DataFrame(bars)
    df['symbol'] = symbol
    return df

In [ ]:
# Fetch daily and intraday data for all symbols
dataframes_daily = []
dataframes_intraday = []

In [ ]:
for symbol in symbols:
    daily_df = fetch_daily_data(symbol)
    intraday_df = fetch_intraday_data(symbol)
    dataframes_daily.append(daily_df)
    dataframes_intraday.append(intraday_df)

In [ ]:
# Combine and preprocess daily data for gap calculations
daily_data = pd.concat(dataframes_daily)
daily_data['date'] = pd.to_datetime(daily_data['date']).dt.tz_localize('UTC').dt.tz_convert('America/New_York')
daily_data.set_index('date', inplace=True)

In [ ]:
# Combine and preprocess intraday data for pre-market calculations
intraday_data = pd.concat(dataframes_intraday)
intraday_data['date'] = pd.to_datetime(intraday_data['date']).dt.tz_localize('UTC').dt.tz_convert('America/New_York')
intraday_data.set_index('date', inplace=True)

In [ ]:
# Pre-Market and Gap Calculations
pre_market_start = pd.Timestamp('04:00:00').time()
pre_market_end = pd.Timestamp('09:00:00').time()
intraday_data['is_pre_market'] = intraday_data.index.time.between(pre_market_start, pre_market_end)

In [ ]:
daily_data['next_open'] = daily_data.groupby('symbol')['open'].shift(-1)
daily_data['gap_pct'] = ((daily_data['next_open'] - daily_data['close']) / daily_data['close']) * 100

In [ ]:
# Pre-Market Volume and Dollar Volume Calculations
pre_market_stats = intraday_data[intraday_data['is_pre_market']].groupby([intraday_data.index.date, 'symbol']).agg(
    pre_market_volume=('volume', 'sum'),
    pre_market_dollar_vol=('volume', lambda x: (x * intraday_data['close']).sum())
).reset_index()
pre_market_stats['date'] = pd.to_datetime(pre_market_stats['date']).dt.tz_localize('UTC').dt.tz_convert('America/New_York')

In [ ]:
combined_data = daily_data.reset_index().merge(pre_market_stats, on=['date', 'symbol'], how='left')

In [ ]:
# Relative Volume Calculation
combined_data['avg_pre_market_vol_30d'] = combined_data.groupby('symbol')['pre_market_volume'].transform(
    lambda x: x.rolling(window=30, min_periods=1).mean())
combined_data['avg_pre_market_dollar_vol_30d'] = combined_data.groupby('symbol')['pre_market_dollar_vol'].transform(
    lambda x: x.rolling(window=30, min_periods=1).mean())
combined_data['rel_vol'] = combined_data['pre_market_volume'] / combined_data['avg_pre_market_vol_30d']
combined_data['rel_dollar_vol'] = combined_data['pre_market_dollar_vol'] / combined_data['avg_pre_market_dollar_vol_30d']

In [ ]:
combined_data['gap_fill'] = (combined_data['low'] <= combined_data['close']).astype(int)

In [ ]:
# Prepare data for modeling
feature_cols = ['rel_vol', 'rel_dollar_vol', 'gap_pct']
X = combined_data[feature_cols]
y = combined_data['gap_fill']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Train model
model = LogisticRegression()
model.fit(X_train, y_train)

In [ ]:
# Evaluate model
y_pred = model.predict(X_test)
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Accuracy:", accuracy_score(y_test, y_pred))

In [ ]:
# Feature importance visualization
coef_df = pd.DataFrame({'Feature': feature_cols, 'Coefficient': model.coef_[0]})
coef_df.sort_values(by='Coefficient', ascending=False, inplace=True)
plt.barh(coef_df['Feature'], coef_df['Coefficient'])
plt.xlabel('Coefficient Value')
plt.title('Feature Importance')
plt.show()

In [ ]:
# Save the model
with open('logistic_regression_model.pkl', 'wb') as f:
    pickle.dump(model, f)

In [ ]:
# Save processed data
combined_data.to_csv('processed_combined_data.csv', index=False)

In [ ]:
# Disconnect IB
ib.disconnect()
print("Model training completed and saved.")